# 1. Setup
**Baseline 1: Independent CNNs.** Two separate CNNs. One predicts the crop, the other predicts the disease.
A pair score is log P(crop) + log P(disease), so unseen pairs get a score too.

In [1]:
# Colab setup: dataset stays private in Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
from pathlib import Path
repo = Path('/content/COS30082-Plant-Disease-OOD')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/COS30082-ML-Project-2026/COS30082-Plant-Disease-OOD.git',str(repo)], check=True)
subprocess.run(['pip','-q','install','timm','scikit-learn','pandas','matplotlib','seaborn','tqdm','py7zr'], check=True)
archive = Path('/content/drive/MyDrive/DPD_Small.7z')
assert archive.exists(), f'Dataset not found: {archive}'
import py7zr
if not (repo/'data'/'DPD_Small_train.csv').exists():
    with py7zr.SevenZipFile(archive, 'r') as z:
        z.extractall('/content/dpd_extracted')
    import shutil
    for filename in ['DPD_Small_train.csv','DPD_Small_test_seen.csv','DPD_Small_test_unseen.csv']:
        matches = list(Path('/content/dpd_extracted').rglob(filename))
        assert len(matches)==1, (filename, matches)
        shutil.copy2(matches[0], repo/'data'/filename)
    image_dirs = list(Path('/content/dpd_extracted').rglob('DPD_Small_dataset'))
    assert len(image_dirs)==1, image_dirs
    (repo/'data'/'DPD_Small_dataset').symlink_to(image_dirs[0], target_is_directory=True)
os.chdir(repo/'baselines')

import copy
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  # lets the notebook import common/

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

from common import config
from common.config import CHECKPOINT_DIR, DEVICE, NUM_CROPS, NUM_DISEASES, set_seed
from common.dataset import get_loaders, get_pairs
from common.evaluate import evaluate, val_accuracy

MODEL_NAME = "baseline1_independent"
DEBUG = True  # True: small data, 1 epoch. Set False for the full run.
HP = {
    "backbone": "resnet50",
    "crop_lr": 1e-4,
    "disease_lr": 3e-4,
    "weight_decay": 1e-4,
    "crop_epochs": 20,
    "disease_epochs": 20,
    "batch_size": 64,
    "optimizer": "AdamW",
    "label_smoothing": 0.1,
}

crop_epochs = 1 if DEBUG else HP["crop_epochs"]
disease_epochs = 1 if DEBUG else HP["disease_epochs"]

# Single-process loading avoids Windows worker memory errors.
config.NUM_WORKERS = 0
HP["batch_size"] = 16 if DEBUG else HP["batch_size"]
HP["debug"] = DEBUG
RUN_NAME = MODEL_NAME + ("_debug" if DEBUG else "")
set_seed(42)
# Full baseline run on Colab GPU
DEBUG = False
HP['debug'] = False
HP['batch_size'] = 64
crop_epochs = HP['crop_epochs']
disease_epochs = HP['disease_epochs']
RUN_NAME = MODEL_NAME
config.NUM_WORKERS = 2
output_root = Path('/content/drive/MyDrive/COS30082-Plant-Disease-OOD/baseline1')
output_root.mkdir(parents=True, exist_ok=True)
config.RESULTS_DIR = output_root / 'results'
config.CHECKPOINT_DIR = output_root / 'checkpoints'
CHECKPOINT_DIR = config.CHECKPOINT_DIR
config.RESULTS_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)
assert torch.cuda.is_available(), 'Select a GPU runtime before full training'
print('GPU:', torch.cuda.get_device_name(0))


Mounted at /content/drive
GPU: Tesla T4


In [2]:
# Loader settings are configured in Setup above.


# 2. Data

In [3]:
train_loader, val_loader, seen_loader, unseen_loader = get_loaders(
    HP["batch_size"], norm="imagenet", debug=DEBUG)
# Reduce host memory pressure before any loader is iterated.
for loader in (train_loader, val_loader, seen_loader, unseen_loader):
    loader.pin_memory = False
pairs = get_pairs().to(DEVICE)

# 3. Model

In [8]:
# Recreate loaders after interrupted execution
train_loader, val_loader, seen_loader, unseen_loader = get_loaders(HP['batch_size'], norm='imagenet', debug=False)
print('Split sizes:', *(len(x.dataset) for x in (train_loader, val_loader, seen_loader, unseen_loader)))
set_seed(42)
def make_classifier(num_classes):
    return timm.create_model(HP["backbone"], pretrained=True, num_classes=num_classes)


crop_model = make_classifier(NUM_CROPS).to(DEVICE)
disease_model = make_classifier(NUM_DISEASES).to(DEVICE)


@torch.no_grad()
def predict(images):
    crop_model.eval()
    disease_model.eval()
    images = images.to(DEVICE)
    crop_log_prob = F.log_softmax(crop_model(images).float(), dim=1)
    disease_log_prob = F.log_softmax(disease_model(images).float(), dim=1)
    # Treat crop and disease as independent, so pair score = sum of log-probs
    return crop_log_prob[:, pairs[:, 0]] + disease_log_prob[:, pairs[:, 1]]

Split sizes: 7223 777 2000 1448


# 4. Training

In [9]:
crop_epochs = 1 if DEBUG else HP["crop_epochs"]
disease_epochs = 1 if DEBUG else HP["disease_epochs"]

crop_optimizer = torch.optim.AdamW(
    crop_model.parameters(),
    lr=HP["crop_lr"],
    weight_decay=HP["weight_decay"],
)
disease_optimizer = torch.optim.AdamW(
    disease_model.parameters(),
    lr=HP["disease_lr"],
    weight_decay=HP["weight_decay"],
)

crop_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    crop_optimizer, T_max=crop_epochs
)
disease_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    disease_optimizer, T_max=disease_epochs
)
criterion = nn.CrossEntropyLoss(label_smoothing=HP["label_smoothing"])
best_val, best_state = -1.0, None
import json, time
from tqdm.auto import tqdm
history = []
start_epoch = 0
resume_path = CHECKPOINT_DIR / "baseline1_latest.pt"
if resume_path.exists():
    saved = torch.load(resume_path, map_location=DEVICE, weights_only=False)
    crop_model.load_state_dict(saved["crop"])
    disease_model.load_state_dict(saved["disease"])
    crop_optimizer.load_state_dict(saved["crop_optimizer"])
    disease_optimizer.load_state_dict(saved["disease_optimizer"])
    crop_scheduler.load_state_dict(saved["crop_scheduler"])
    disease_scheduler.load_state_dict(saved["disease_scheduler"])
    start_epoch = saved["epoch"]
    best_val, best_state = saved["best_val"], saved["best_state"]
    history = saved["history"]
    print("Resuming at epoch", start_epoch + 1)

for epoch in range(start_epoch, max(crop_epochs, disease_epochs)):
    train_crop = epoch < crop_epochs
    train_disease = epoch < disease_epochs

    crop_model.train(train_crop)
    disease_model.train(train_disease)

    for images, crop_labels, disease_labels, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/20"):

        images = images.to(DEVICE)
        crop_labels = crop_labels.to(DEVICE)
        disease_labels = disease_labels.to(DEVICE)

        if train_crop:
            crop_optimizer.zero_grad(set_to_none=True)
            crop_loss = criterion(crop_model(images), crop_labels)
            crop_loss.backward()
            crop_optimizer.step()

        if train_disease:
            disease_optimizer.zero_grad(set_to_none=True)
            disease_loss = criterion(disease_model(images), disease_labels)
            disease_loss.backward()
            disease_optimizer.step()

    if train_crop:
        crop_scheduler.step()
    if train_disease:
        disease_scheduler.step()

    val_top1 = val_accuracy(predict, val_loader)

    print(f"epoch {epoch + 1}: crop loss {crop_loss.item():.3f}, "
          f"disease loss {disease_loss.item():.3f}, val top-1 {val_top1:.2f}%")
    if val_top1 > best_val:  # keep the best epoch by val, never by test
        best_val = val_top1
        best_state = copy.deepcopy({"crop": crop_model.state_dict(), "disease": disease_model.state_dict()})

    history.append({"epoch": epoch+1, "val_top1": val_top1, "crop_loss_last_batch": crop_loss.item(), "disease_loss_last_batch": disease_loss.item()})
    torch.save({"epoch": epoch+1, "crop": crop_model.state_dict(), "disease": disease_model.state_dict(), "crop_optimizer": crop_optimizer.state_dict(), "disease_optimizer": disease_optimizer.state_dict(), "crop_scheduler": crop_scheduler.state_dict(), "disease_scheduler": disease_scheduler.state_dict(), "best_val": best_val, "best_state": best_state, "history": history}, resume_path)
    torch.save({"model_name": RUN_NAME, "state_dict": best_state, "hp": HP}, CHECKPOINT_DIR / f"{RUN_NAME}.pt")
    (config.RESULTS_DIR / "baseline1_training_history.json").write_text(json.dumps(history, indent=2))

crop_model.load_state_dict(best_state["crop"])
disease_model.load_state_dict(best_state["disease"])

epoch 1: crop loss 3.200, disease loss 3.278, val top-1 22.65%


epoch 2: crop loss 2.349, disease loss 2.309, val top-1 44.92%


epoch 3: crop loss 1.827, disease loss 2.177, val top-1 58.82%


epoch 4: crop loss 1.378, disease loss 1.724, val top-1 67.57%


epoch 5: crop loss 1.412, disease loss 1.551, val top-1 70.79%


...

epoch 18: crop loss 0.879, disease loss 0.972, val top-1 81.85%
epoch 18: crop loss 0.879, disease loss 0.972, val top-1 81.85%


epoch 19: crop loss 0.864, disease loss 0.958, val top-1 82.37%
epoch 19: crop loss 0.864, disease loss 0.958, val top-1 82.37%


epoch 20: crop loss 0.850, disease loss 0.947, val top-1 81.98%
epoch 20: crop loss 0.850, disease loss 0.947, val top-1 81.98%


<All keys matched successfully>

<All keys matched successfully>

# 5. Evaluation

In [10]:
results = evaluate(predict, seen_loader, unseen_loader, RUN_NAME, HP, robust=True)

{
  "model": "baseline1_independent",
  "seen_top1": 62.1,
  "seen_top5": 84.15,
  "unseen_top1": 8.49,
  "unseen_top5": 38.26,
  "harmonic_mean": 14.94,
  "crop_acc": 85.75,
  "disease_acc": 64.95,
  "robustness": {
    "brightness": {
      "top1": 61.25,
      "drop": 0.85
    },
    "blur": {
      "top1": 38.5,
      "drop": 23.6
    },
    "noise": {
      "top1": 36.35,
      "drop": 25.75
    }
  }
}
{
  "model": "baseline1_independent",
  "seen_top1": 62.1,
  "seen_top5": 84.15,
  "unseen_top1": 8.49,
  "unseen_top5": 38.26,
  "harmonic_mean": 14.94,
  "crop_acc": 85.75,
  "disease_acc": 64.95,
  "robustness": {
    "brightness": {
      "top1": 61.25,
      "drop": 0.85
    },
    "blur": {
      "top1": 38.5,
      "drop": 23.6
    },
    "noise": {
      "top1": 36.35,
      "drop": 25.75
    }
  }
}


# 6. Save

In [ ]:
CHECKPOINT_DIR.mkdir(exist_ok=True)
torch.save({"model_name": RUN_NAME, "state_dict": best_state, "hp": HP},
           CHECKPOINT_DIR / f"{RUN_NAME}.pt")